# RankResume - Document to PostgreSQL Pipeline

This notebook runs local document extraction, structured JSON parsing, validation, PostgreSQL JSONB storage, and dataset export. It does not implement matching or ranking.

## 1. Install dependencies if needed

In [ ]:
# Run once in the notebook kernel if imports are unavailable.
# %pip install pdfplumber python-docx psycopg2-binary skill-extractor



## 2. Import the modular pipeline

In [1]:
import sys
import importlib
from pathlib import Path

DB_STORE = Path.cwd()
if not (DB_STORE / 'rankresume_pipeline.py').is_file():
    candidates = [DB_STORE / 'DB_Store', DB_STORE / 'Project' / 'DB_Store']
    DB_STORE = next(path for path in candidates if (path / 'rankresume_pipeline.py').is_file())
if str(DB_STORE) not in sys.path:
    sys.path.insert(0, str(DB_STORE))
import rankresume_pipeline
importlib.reload(rankresume_pipeline)
from rankresume_pipeline import *

print('Configured resumes path:', RESUMES_DIR)
print('Configured JDs path:', JDS_DIR)
print('Resume extensions:', sorted(RESUME_EXTENSIONS))
print('JD extensions:', sorted(JD_EXTENSIONS))

Configured resumes path: C:\Python_ML\Project\Data\resumes
Configured JDs path: C:\Python_ML\Project\Data\jds
Resume extensions: ['.doc', '.docx', '.pdf']
JD extensions: ['.csv', '.doc', '.docx', '.json', '.pdf']


## 3. Verify input folders and local readers

In [2]:
resume_dir, jd_dir = resolve_input_dirs()
print('Using resumes:', resume_dir)
print('Using JDs:', jd_dir)
print('Resume files:', len(files_in(resume_dir)))
print('JD files:', len(files_in(jd_dir)))
print('pdfplumber reader: available')
print('python-docx reader: available')
print('DOC reader: antiword or LibreOffice is checked only when a .doc file is encountered')

Using resumes: C:\Python_ML\Project\Data\resumes
Using JDs: C:\Python_ML\Project\Data\jds
Resume files: 19
JD files: 3
pdfplumber reader: available
python-docx reader: available
DOC reader: antiword or LibreOffice is checked only when a .doc file is encountered


## 5. PostgreSQL safety check and schema setup

`RESET_TABLES` is deliberately false. The pipeline scans all supported files, checks each filename before extraction, inserts only new documents, and never deletes database rows when source files are removed.

In [3]:
sample = 'C++ C# .NET ASP.NET Node.js React.js Next.js CI/CD Spring Boot REST API PostgreSQL Machine Learning JavaScript Kubernetes Apache Kafka'
print('Technology aliases:', normalize_skills(['C++', 'cpp', 'c plus plus', 'JavaScript', 'js', 'Machine Learning', 'ml']))
print('Discovered skills:', discover_skills(sample))
print('Cleaned sample:', clean_text(sample))
print('Extraction helpers are available; process_batch performs duplicate checks before extracting any existing file.')

Technology aliases: ['c++', 'javascript', 'machine learning']


C:\Users\Prapti\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-29 18:15:11,087 INFO: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-09-29 18:15:11,412 INFO: HTTP Request: HEAD https://huggingface.co/Xenova/all-MiniLM-L6-v2/resolve/main/onnx/model.onnx "HTTP/1.1 302 Found"
2026-09-29 18:15:11,420 WARNING: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-29 18:15:11,691 INFO: HTTP Request: HEAD https://huggingface.co/Xenova/all-MiniLM-L6-v2/resolve/main/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
2026-09-29 18:15:11,733 INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Xenova/all-MiniLM-L6-v2/751bff37182d3f1213fa05d7196

Discovered skills: ['c++', 'c#', '.net', 'node.js', 'react.js', 'next.js', 'ci/cd', 'spring boot', 'restful api', 'postgresql', 'machine learning', 'javascript', 'apache kafka', 'kubernetes']
Cleaned sample: c++ c# .net .net node.js react.js next.js ci/cd spring boot rest api postgresql machine learning javascript kubernetes apache kafka
Extraction helpers are available; process_batch performs duplicate checks before extracting any existing file.


## 5. PostgreSQL safety check and schema setup

`RESET_TABLES` is deliberately false. The pipeline synchronizes application rows to files currently present in the Data folders and preserves the existing database credentials and JSON schema.

In [4]:
RESET_TABLES = False
connection = connect_db()
print('Verified database:', DB_NAME)
print('Existing public tables:', inspect_tables(connection))
create_schema(connection, reset=RESET_TABLES)
seed_dictionary(connection)
esco_csv = find_esco_skills_csv()
esco_count = import_esco_skills(connection, esco_csv)
load_esco_labels(connection)
print('ESCO CSV:', esco_csv)
print('ESCO records:', esco_count)
print('Schema, technology aliases, and ESCO reference data are ready.')

Verified database: rankresume
Existing public tables: ['ESCO_Skills', 'ats_score', 'candidate', 'job_description', 'technology_alias', 'technology_dictionary']


2026-09-29 18:15:17,444 INFO: Imported 13939 ESCO skills from C:\Python_ML\Project\skills_en.csv


ESCO CSV: C:\Python_ML\Project\skills_en.csv
ESCO records: 13939
Schema, technology aliases, and ESCO reference data are ready.


## 6. Run the complete batch

Each document is handled independently. A failed or unsupported file is logged and does not stop the remaining batch.

In [5]:
summary = process_batch(connection, reset=False)
print_summary(summary)
print('\n========== SAMPLE RESUME JSON ==========')
resume_dataset = json.loads((DATA_ROOT / 'resumes.json').read_text(encoding='utf-8'))
print(json.dumps(resume_dataset['resumes'][0], indent=2, ensure_ascii=False)[:3000])
print('\n========== SAMPLE JD JSON ==========')
jd_dataset = json.loads((DATA_ROOT / 'jds.json').read_text(encoding='utf-8'))
print(json.dumps(jd_dataset['job_descriptions'][0], indent=2, ensure_ascii=False)[:3000])

2026-09-29 18:15:20,863 INFO: Imported 13939 ESCO skills from C:\Python_ML\Project\skills_en.csv
2026-09-29 18:15:21,510 INFO: Scanning resumes directory: found 19 supported files
2026-09-29 18:15:21,511 INFO: Scanning JDs directory: found 3 supported files
2026-09-29 18:15:21,511 INFO: [1/19] Checking resume: Aarya_Kale.pdf
2026-09-29 18:15:21,560 INFO: Already exists in database. Skipping: Aarya_Kale.pdf
2026-09-29 18:15:21,560 INFO: [2/19] Checking resume: Abhijeet.pdf
2026-09-29 18:15:21,560 INFO: Already exists in database. Skipping: Abhijeet.pdf
2026-09-29 18:15:21,560 INFO: [3/19] Checking resume: Adity_Gowari.pdf
2026-09-29 18:15:21,563 INFO: Already exists in database. Skipping: Adity_Gowari.pdf
2026-09-29 18:15:21,564 INFO: [4/19] Checking resume: Aishwarya_Khedkar.pdf
2026-09-29 18:15:21,564 INFO: Already exists in database. Skipping: Aishwarya_Khedkar.pdf
2026-09-29 18:15:21,566 INFO: [5/19] Checking resume: Aniket_Jadhav_Resume.pdf
2026-09-29 18:15:21,567 INFO: Already exi

INGESTION SUMMARY

Resumes:
    Files found       : 19
    Already in DB     : 19
    Newly inserted    : 0
    Failed            : 0

JDs:
    Files found       : 3
    Already in DB     : 3
    Newly inserted    : 0
    Failed            : 0

Total files checked  : 22
Total new records    : 0
Total skipped        : 22
Total failed         : 0

Database: rankresume
Candidate records: 19
Job Description records: 3

Resume JSON validation: 0 passed, 0 flagged
JD JSON validation: 0 passed, 0 flagged
Datasets: C:\Python_ML\Project\Data\resumes.json and C:\Python_ML\Project\Data\jds.json
ESCO records: 13939

========== SAMPLE RESUME JSON ==========
{
  "resume_file_name": "Aarya_Kale.pdf",
  "candidate_name": "AARYA KALE",
  "email": "kaleaarya704@gmail.com",
  "phone": "8421749236",
  "structured_data": {
    "skill": [
      "html5",
      "css3",
      "react.js",
      "restful api",
      "java",
      "c++",
      "spring boot",
      "data structures",
      "object-oriented program

## 7. Validate stored values

In [6]:
print('Running complete parser and database validation...')
resume_results = []
for path in files_in(resume_dir):
    if path.suffix.lower() in RESUME_EXTENSIONS:
        parsed = parse_resume(extract_text(path))
        errors = validate_resume_json(parsed)
        resume_results.append((path.name, errors))
        assert not errors, (path.name, errors)

jd_results = []
for path in files_in(jd_dir):
    if path.suffix.lower() in JD_EXTENSIONS:
        raw, source_title = extract_jd_text(path)
        parsed = parse_jd(raw, source_title or extract_job_title(raw))
        errors = validate_jd_json(parsed)
        jd_results.append((path.name, errors))
        assert not errors, (path.name, errors)

assert resume_results, 'No supported resume files were found'
assert jd_results, 'No supported JD files were found'
assert parse_jd('Minimum 60% marks required.', 'test')['expected_education'][0]['percentage/mark'] == '60%'
assert parse_jd('Candidates should have at least 60 percent.', 'test')['expected_education'][0]['percentage/mark'] == '60%'
assert parse_jd('Achieved 90% improvement in performance.', 'test')['expected_education'] == []
with connection.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM job_description WHERE structured_data ? 'expected_extra_skills'")
    legacy_jd_count = cursor.fetchone()[0]
print('Real resume files validated:', len(resume_results))
print('Real JD files validated:', len(jd_results))
print('Stored JD legacy-field count:', legacy_jd_count)
print('Stored JSONB validation: PASS')

Running complete parser and database validation...


2026-09-29 18:15:27,288 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:28,051 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:29,109 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:31,072 INFO: Resume extraction counts: projects=3 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:31,607 INFO: Resume extraction counts: projects=2 certificates_or_achievements=3 experience_entries=0
2026-09-29 18:15:32,009 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:32,356 INFO: Resume extraction counts: projects=3 certificates_or_achievements=3 experience_entries=0
2026-09-29 18:15:33,044 INFO: Resume extraction counts: projects=2 certificates_or_achievements=1 experience_entries=0
2026-09-29 18:15:34,128 INFO: Resume extraction 

Real resume files validated: 19
Real JD files validated: 3
Stored JD legacy-field count: 0
Stored JSONB validation: PASS


In [7]:
print('Targeted parser validation')

sample_resume_paths = [
    path for path in files_in(resume_dir)
    if path.suffix.lower() in RESUME_EXTENSIONS
][:2]
assert sample_resume_paths, 'No supported resume files were found'
for path in sample_resume_paths:
    parsed = parse_resume(extract_text(path))
    assert not validate_resume_json(parsed), path.name
    print(path.name, 'resume JSON: PASS')

for name in ('jd_java_developer.pdf', 'JD Intern_SpringCT.pdf'):
    path = jd_dir / name
    if not path.is_file():
        print(name, 'not present; skipped')
        continue
    raw, source_title = extract_jd_text(path)
    jd = parse_jd(raw, source_title or extract_job_title(raw))
    assert set(jd) == {'job_title', 'expected_skills', 'expected_education', 'expected_experience', 'expected_certificate'}
    assert 'expected_extra_skills' not in jd
    print(name, 'JD JSON structure: PASS')

print('Targeted parser validation: PASS')

Targeted parser validation


2026-09-29 18:15:40,470 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0


Aarya_Kale.pdf resume JSON: PASS


2026-09-29 18:15:41,065 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:41,225 INFO: JD extraction counts: academic_mark=60% education=1 certificates=0


Abhijeet.pdf resume JSON: PASS
jd_java_developer.pdf JD JSON structure: PASS


2026-09-29 18:15:41,800 INFO: JD extraction counts: academic_mark=60% education=1 certificates=0


JD Intern_SpringCT.pdf JD JSON structure: PASS
Targeted parser validation: PASS


In [8]:
selected_resume_names = [path.name for path in sample_resume_paths]
legacy_jds = []
with connection.cursor() as cursor:
    cursor.execute("SELECT jd_file_name FROM job_description WHERE structured_data ? 'expected_extra_skills'")
    legacy_jds = [row[0] for row in cursor.fetchall()]

migration_result = reprocess_existing_documents(
    connection,
    resume_names=selected_resume_names,
    jd_names=legacy_jds,
)
print('Controlled reprocessing:', migration_result)
assert migration_result['resumes'] == len(selected_resume_names)
assert migration_result['jds'] == len(legacy_jds)

2026-09-29 18:15:44,350 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:45,063 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:45,155 INFO: Controlled document reprocessing complete: {'resumes': 2, 'jds': 0}


Controlled reprocessing: {'resumes': 2, 'jds': 0}


In [9]:
selected_resume_names = [path.name for path in sample_resume_paths]
legacy_jds = []
with connection.cursor() as cursor:
    cursor.execute("SELECT jd_file_name FROM job_description WHERE structured_data ? 'expected_extra_skills'")
    legacy_jds = [row[0] for row in cursor.fetchall()]

migration_result = reprocess_existing_documents(
    connection,
    resume_names=selected_resume_names,
    jd_names=legacy_jds,
)
print('Controlled reprocessing:', migration_result)
assert migration_result['resumes'] == len(selected_resume_names)
assert migration_result['jds'] == len(legacy_jds)

2026-09-29 18:15:47,324 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:47,970 INFO: Resume extraction counts: projects=2 certificates_or_achievements=2 experience_entries=0
2026-09-29 18:15:48,063 INFO: Controlled document reprocessing complete: {'resumes': 2, 'jds': 0}


Controlled reprocessing: {'resumes': 2, 'jds': 0}
